# Prepare canonical SciSciNet-v2 field memberships

## Purpose

This notebook prepares the canonical SciSciNet-v2 field memberships used by
the manuscript analysis.

It takes the paper-level analysis file produced by the BigQuery preprocessing
pipeline:

```text
data/disruption_analysis.csv
```
and reconstructs SciSciNet-v2 Level-0 and Level-1 field memberships for those
papers.

The manuscript field analysis uses the 19 canonical Level-0 fields and retains
all Level-0 memberships for multidisciplinary papers.

## Main output
```text
data/sciscinet_field_reconstruction_v2/
    analysis_level0_field_memberships.parquet
```
with columns:
```text
paperid
level_0_field_name
```
A paper can appear more than once because SciSciNet field membership is
multi-membership rather than an exclusive classification.
This output is consumed directly by 03_manuscript_analysis.ipynb.

## 0. Dependencies and SciSciNet-v2 access

SciSciNet-v2 is hosted on Hugging Face and currently requires accepting the
dataset access conditions.

Before running this notebook:

1. Generate `data/disruption_analysis.csv` using the repository's BigQuery
   preprocessing pipeline.
2. Accept the SciSciNet-v2 dataset conditions while signed into Hugging Face.
3. Either set the `HF_TOKEN` environment variable to a Hugging Face read token
   or enter the token when prompted by this notebook.

The notebook downloads and caches:

- `sciscinet_fields.parquet`
- `sciscinet_paperfields.parquet`

under:

```text
data/sciscinet_v2_public/
```
Already downloaded files are reused automatically.

In [ ]:
%pip install -U polars pyarrow pandas huggingface_hub hf_xet psutil duckdb

## 1. Configuration

In [ ]:
import os
os.environ.setdefault("POLARS_MAX_THREADS", "19")

from pathlib import Path
from time import perf_counter
import json
import platform
import sys
from getpass import getpass

import pandas as pd
import duckdb
import polars as pl
import psutil

from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.errors import HfHubHTTPError, GatedRepoError

DATA_PATH = Path("data/disruption_analysis.csv")

SOURCE_DIR = Path("data/sciscinet_v2_public")
OUTPUT_DIR = Path("data/sciscinet_field_reconstruction_v2")

SOURCE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_ID = "Northwestern-CSSI/sciscinet-v2"
REQUESTED_REVISION = "main"

# Authentication:
# Prefer HF_TOKEN from the environment; otherwise prompt securely at runtime.
# Never hard-code a token into this notebook.
HF_TOKEN = os.environ.get("HF_TOKEN")

FIELDS_FILENAME = "sciscinet_fields.parquet"
PAPER_FIELDS_FILENAME = "sciscinet_paperfields.parquet"

FIELDS_PATH = SOURCE_DIR / FIELDS_FILENAME
PAPER_FIELDS_PATH = SOURCE_DIR / PAPER_FIELDS_FILENAME

TARGET_PAPERS_CACHE = OUTPUT_DIR / "analysis_target_papers_nonnull.parquet"
MATCHED_LINKS_CACHE = OUTPUT_DIR / "analysis_matched_sciscinet_v2_fields.parquet"

LEVEL0_LONG = OUTPUT_DIR / "analysis_level0_field_memberships.parquet"
LEVEL1_LONG = OUTPUT_DIR / "analysis_level1_field_memberships.parquet"
PAPER_ARRAYS = OUTPUT_DIR / "analysis_paper_field_arrays.parquet"

LEVEL0_LOOKUP_CSV = OUTPUT_DIR / "level0_field_lookup.csv"
LEVEL1_LOOKUP_CSV = OUTPUT_DIR / "level1_field_lookup.csv"
LEVEL0_COUNTS_CSV = OUTPUT_DIR / "level0_membership_counts.csv"
MULTIPLICITY_CSV = OUTPUT_DIR / "level0_memberships_per_paper.csv"
SCALAR_DIAGNOSTIC_CSV = OUTPUT_DIR / "scalar_field_name_diagnostic.csv"
MANIFEST_JSON = OUTPUT_DIR / "field_reconstruction_manifest.json"

FORCE_DOWNLOAD = False
FORCE_REBUILD_TARGET = False
FORCE_REBUILD_MATCHED_LINKS = False
FORCE_REBUILD_OUTPUTS = False

# DuckDB is used for the large paper-field join and may spill to disk.
DUCKDB_MEMORY_LIMIT = "32GB"
DUCKDB_TEMP_DIR = OUTPUT_DIR / "duckdb_temp"
DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)
DUCKDB_THREADS = 19

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Polars:", pl.__version__)
print("Polars thread pool:", pl.thread_pool_size())
print("Logical CPUs:", os.cpu_count())
print("RAM (GB):", round(psutil.virtual_memory().total / 1024**3, 2))
print()
print("Analysis CSV:", DATA_PATH.resolve())
print("SciSciNet-v2 source directory:", SOURCE_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Analysis CSV not found:\n{DATA_PATH.resolve()}")

## 2. Verify the analysis paper-ID system

This should show OpenAlex-style `W...` identifiers.

The notebook keeps them as strings throughout. It never strips the `W` prefix
and never casts paper IDs to integers.

In [ ]:
analysis_id_preview = (
    pl.scan_csv(
        DATA_PATH,
        infer_schema_length=100_000,
    )
    .select(pl.col("paperid").cast(pl.String))
    .head(10)
    .collect()
)

display(analysis_id_preview.to_pandas())

preview_ids = analysis_id_preview["paperid"].to_list()

if not all(
    isinstance(x, str) and x.startswith("W")
    for x in preview_ids
    if x is not None
):
    raise ValueError(
        "Expected OpenAlex/SciSciNet-v2 paper IDs beginning with 'W'. "
        f"Observed preview: {preview_ids}"
    )

print("PASS: analysis paper IDs are OpenAlex/SciSciNet-v2 style strings.")

## 3. Authenticate and verify SciSciNet-v2 gated access

A valid Hugging Face token is necessary, but the account behind the token must
also be authorized for `Northwestern-CSSI/sciscinet-v2`.

This cell does four things before any large download:

1. uses the `HF_TOKEN` environment variable if present;
2. otherwise prompts securely with `getpass()` so the token is not echoed;
3. verifies the token and prints the authenticated HF account;
4. checks that this account can access SciSciNet-v2.

The token is never written to the notebook outputs or manifest.

In [ ]:
if not HF_TOKEN:
    HF_TOKEN = getpass(
        "Paste a Hugging Face READ token (input hidden): "
    ).strip()

if not HF_TOKEN:
    raise RuntimeError(
        "No Hugging Face token supplied. Set HF_TOKEN or rerun this cell."
    )

auth_api = HfApi(token=HF_TOKEN)

try:
    who = auth_api.whoami()
except Exception as e:
    raise RuntimeError(
        "The Hugging Face token could not be authenticated."
    ) from e

hf_username = (
    who.get("name")
    or who.get("fullname")
    or who.get("email")
    or "<unknown>"
)

print("Authenticated Hugging Face account:", hf_username)
print("Token authentication: PASS")

try:
    gated_info = auth_api.dataset_info(
        REPO_ID,
        revision=REQUESTED_REVISION,
    )
except GatedRepoError as e:
    print("\n" + "=" * 78)
    print("SCISCINET-v2 ACCESS HAS NOT BEEN GRANTED TO THIS ACCOUNT")
    print("=" * 78)
    print("Authenticated account:", hf_username)
    print()
    print(
        "The token is valid, but Hugging Face reports that this account "
        "is not on the authorized list for Northwestern-CSSI/sciscinet-v2."
    )
    print()
    print("Open, while signed into THIS SAME account:")
    print(
        "  https://huggingface.co/datasets/Northwestern-CSSI/sciscinet-v2"
    )
    print()
    print("Request/accept access there, then rerun this cell.")
    print(
        "Creating another token will not bypass the gate unless the account "
        "it belongs to has dataset access."
    )
    print("=" * 78)

    raise RuntimeError(
        "Valid HF token, but SciSciNet-v2 gated access is not granted."
    ) from e

except HfHubHTTPError as e:
    raise RuntimeError(
        "Hugging Face authentication succeeded, but the SciSciNet-v2 "
        "access check failed for another HTTP reason."
    ) from e

RESOLVED_REVISION = gated_info.sha

print("\nSciSciNet-v2 gated access: PASS")
print("Resolved revision:", RESOLVED_REVISION)

## 4. Download the two SciSciNet-v2 field files

If you obtain the files manually after access is granted, place them at:

```text
data\sciscinet_v2_public\sciscinet_fields.parquet
data\sciscinet_v2_public\sciscinet_paperfields.parquet
```

The download cell will reuse them automatically.


In [ ]:
def get_sciscinet_v2_file(filename, expected_path, force=False):
    expected_path = Path(expected_path)

    if expected_path.exists() and not force:
        print(
            f"Reusing {filename}: "
            f"{expected_path.stat().st_size / 1024**3:.3f} GB"
        )
        return expected_path

    print(f"Downloading {filename}")
    print(f"  repository: {REPO_ID}")
    print(f"  revision:   {RESOLVED_REVISION}")
    print(f"  account:    {hf_username}")

    try:
        downloaded = hf_hub_download(
            repo_id=REPO_ID,
            filename=filename,
            repo_type="dataset",
            revision=RESOLVED_REVISION,
            local_dir=SOURCE_DIR,
            force_download=force,
            token=HF_TOKEN,
        )

    except GatedRepoError as e:
        raise RuntimeError(
            "The token is valid, but this Hugging Face account does not "
            "currently have SciSciNet-v2 gated access. Request/accept access "
            "at https://huggingface.co/datasets/Northwestern-CSSI/sciscinet-v2 "
            "while signed into the same account, then rerun."
        ) from e

    except HfHubHTTPError as e:
        raise RuntimeError(
            f"Hugging Face HTTP error while downloading {filename}."
        ) from e

    downloaded = Path(downloaded)

    print(
        "Ready:",
        downloaded.resolve(),
        f"({downloaded.stat().st_size / 1024**3:.3f} GB)",
    )

    return downloaded


FIELDS_PATH = get_sciscinet_v2_file(
    FIELDS_FILENAME,
    FIELDS_PATH,
    FORCE_DOWNLOAD,
)

PAPER_FIELDS_PATH = get_sciscinet_v2_file(
    PAPER_FIELDS_FILENAME,
    PAPER_FIELDS_PATH,
    FORCE_DOWNLOAD,
)

## 5. Inspect the public v2 schemas before any large join

This cell prints the actual public Parquet schemas and resolves the required
columns case-insensitively.

Expected semantics:

### Fields
- field ID
- display name
- level

### Paper fields
- paper ID
- field ID

If the public schema changes, this cell fails immediately rather than beginning
the 10-GB scan.

In [ ]:
fields_schema = pl.scan_parquet(FIELDS_PATH).collect_schema()
paper_fields_schema = pl.scan_parquet(PAPER_FIELDS_PATH).collect_schema()

print("=== sciscinet_fields.parquet schema ===")
for name, dtype in fields_schema.items():
    print(f"{name:35s} {dtype}")

print("\n=== sciscinet_paperfields.parquet schema ===")
for name, dtype in paper_fields_schema.items():
    print(f"{name:35s} {dtype}")


def resolve_column(schema, aliases, semantic_name):
    original = list(schema.names())
    lower_to_original = {c.lower(): c for c in original}

    for alias in aliases:
        if alias.lower() in lower_to_original:
            return lower_to_original[alias.lower()]

    raise ValueError(
        f"Could not find a column for {semantic_name!r}.\n"
        f"Tried aliases: {aliases}\n"
        f"Available columns: {original}"
    )


F_FIELD_ID = resolve_column(
    fields_schema,
    ["fieldid", "field_id", "id"],
    "field ID in fields",
)

F_DISPLAY_NAME = resolve_column(
    fields_schema,
    ["display_name", "field_name", "name"],
    "field display name",
)

F_LEVEL = resolve_column(
    fields_schema,
    ["level", "field_level"],
    "field level",
)

PF_PAPER_ID = resolve_column(
    paper_fields_schema,
    ["paperid", "paper_id"],
    "paper ID in paper-field links",
)

PF_FIELD_ID = resolve_column(
    paper_fields_schema,
    ["fieldid", "field_id"],
    "field ID in paper-field links",
)

print("\nResolved columns:")
print(" fields.fieldid      ->", F_FIELD_ID)
print(" fields.display_name ->", F_DISPLAY_NAME)
print(" fields.level        ->", F_LEVEL)
print(" paperfields.paperid ->", PF_PAPER_ID)
print(" paperfields.fieldid ->", PF_FIELD_ID)

## 6. Validate the SciSciNet-v2 field dictionary

Current SciSciNet-v2 documentation reports 303 field records overall. The
manuscript requires **exactly 19 Level-0 fields**.

We require 19 Level-0 fields and report the Level-1 count from the pinned public
release rather than assuming the v1 count.

In [ ]:
fields = (
    pl.read_parquet(FIELDS_PATH)
    .select([
        pl.col(F_FIELD_ID).cast(pl.String).alias("fieldid"),
        pl.col(F_DISPLAY_NAME).cast(pl.String).alias("display_name"),
        pl.col(F_LEVEL).cast(pl.Int32, strict=False).alias("level"),
    ])
    .filter(pl.col("level").is_in([0, 1]))
    .unique()
)

print("Fields retained at levels 0/1:", fields.height)

level_counts = (
    fields
    .group_by("level")
    .agg([
        pl.len().alias("rows"),
        pl.col("fieldid").n_unique().alias("unique_fieldids"),
        pl.col("display_name").n_unique().alias("unique_names"),
    ])
    .sort("level")
)

display(level_counts.to_pandas())

n_level0 = (
    fields
    .filter(pl.col("level") == 0)
    .select(pl.col("fieldid").n_unique())
    .item()
)

n_level1 = (
    fields
    .filter(pl.col("level") == 1)
    .select(pl.col("fieldid").n_unique())
    .item()
)

assert n_level0 == 19, (
    f"Expected exactly 19 Level-0 fields, found {n_level0}."
)

level0_lookup = (
    fields
    .filter(pl.col("level") == 0)
    .select([
        pl.col("fieldid").alias("field_id"),
        pl.col("display_name").alias("level_0_field_name"),
    ])
    .unique()
    .sort("level_0_field_name")
)

level1_lookup = (
    fields
    .filter(pl.col("level") == 1)
    .select([
        pl.col("fieldid").alias("field_id"),
        pl.col("display_name").alias("level_1_field_name"),
    ])
    .unique()
    .sort("level_1_field_name")
)

level0_lookup.write_csv(LEVEL0_LOOKUP_CSV)
level1_lookup.write_csv(LEVEL1_LOOKUP_CSV)

print("\nCanonical Level-0 fields:")
display(level0_lookup.to_pandas())

## 7. Cache the analysis paper IDs

The field reconstruction only needs paper IDs from the main analysis sample.
Paper IDs are retained as OpenAlex/SciSciNet-v2 strings throughout.

In [ ]:
def parquet_valid(path, required_columns):
    path = Path(path)

    if not path.exists() or path.stat().st_size == 0:
        return False

    try:
        schema = pl.scan_parquet(path).collect_schema()
        return set(required_columns).issubset(set(schema.names()))
    except Exception:
        return False


def remove_invalid_parquet(path, required_columns):
    path = Path(path)

    if path.exists() and not parquet_valid(path, required_columns):
        print("Removing incomplete/incompatible file:", path.resolve())
        try:
            path.unlink()
        except FileNotFoundError:
            pass


remove_invalid_parquet(
    TARGET_PAPERS_CACHE,
    {"paperid", "scalar_field_name"},
)

if FORCE_REBUILD_TARGET and TARGET_PAPERS_CACHE.exists():
    TARGET_PAPERS_CACHE.unlink()

if not parquet_valid(
    TARGET_PAPERS_CACHE,
    {"paperid", "scalar_field_name"},
):
    t0 = perf_counter()

    print("Building target-paper cache from:")
    print(" ", DATA_PATH.resolve())

    target_lf = (
        pl.scan_csv(
            DATA_PATH,
            infer_schema_length=100_000,
            ignore_errors=False,
        )
        .select([
            pl.col("paperid").cast(pl.String).alias("paperid"),
            pl.col("field_name").cast(pl.String).alias("scalar_field_name"),
        ])
        # The analysis export contains exactly one row with a null paperid.
        # It cannot be joined to SciSciNet, so exclude null/blank IDs here.
        .filter(
            pl.col("paperid").is_not_null()
            & (pl.col("paperid").str.len_chars() > 0)
        )
    )

    try:
        target_lf.sink_parquet(
            TARGET_PAPERS_CACHE,
            compression="zstd",
            compression_level=3,
            statistics=True,
            maintain_order=False,
        )
    except Exception:
        if TARGET_PAPERS_CACHE.exists():
            try:
                TARGET_PAPERS_CACHE.unlink()
            except Exception:
                pass
        raise

    print(
        f"Target-paper cache built in "
        f"{(perf_counter() - t0) / 60:.2f} minutes."
    )
else:
    print("Reusing:", TARGET_PAPERS_CACHE.resolve())


target_audit = (
    pl.scan_parquet(TARGET_PAPERS_CACHE)
    .select([
        pl.len().alias("rows"),
        pl.col("paperid").n_unique().alias("unique_paperids"),
        pl.col("paperid").null_count().alias("null_paperids"),
        pl.col("scalar_field_name").n_unique().alias("unique_scalar_fields"),
    ])
    .collect()
)

display(target_audit.to_pandas())

target_stats = target_audit.row(0, named=True)

assert target_stats["rows"] == target_stats["unique_paperids"], (
    "Expected exactly one row per non-null paper ID after filtering."
)
assert target_stats["null_paperids"] == 0

raw_id_audit = (
    pl.scan_csv(
        DATA_PATH,
        infer_schema_length=100_000,
    )
    .select([
        pl.len().alias("raw_rows"),
        pl.col("paperid").null_count().alias("raw_null_paperids"),
    ])
    .collect()
    .row(0, named=True)
)

print(
    f"Validated {target_stats['unique_paperids']:,} "
    "unique non-null OpenAlex/SciSciNet-v2 paper IDs."
)
print(
    "Excluded raw rows with null paperid:",
    raw_id_audit["raw_null_paperids"],
)

if raw_id_audit["raw_null_paperids"] != 1:
    print(
        "NOTE: the manuscript analysis export had exactly one null "
        "paperid; this count has changed."
    )

## 8. Confirm public paper IDs are compatible

This reads only a tiny prefix of `sciscinet_paperfields.parquet`.

Both sides should look like `W...` strings.

In [ ]:
public_id_preview = (
    pl.scan_parquet(PAPER_FIELDS_PATH)
    .select(
        pl.col(PF_PAPER_ID).cast(pl.String).alias("paperid")
    )
    .head(10)
    .collect()
)

display(public_id_preview.to_pandas())

public_preview_ids = public_id_preview["paperid"].to_list()

if not all(
    isinstance(x, str) and x.startswith("W")
    for x in public_preview_ids
    if x is not None
):
    raise ValueError(
        "Public SciSciNet-v2 paper IDs do not look like expected "
        f"OpenAlex W... IDs. Observed: {public_preview_ids}"
    )

print("PASS: both datasets use compatible OpenAlex W... paper IDs.")

## 9. Reconstruct canonical paper-field links with DuckDB

This is the largest join in the field-preparation workflow.

DuckDB joins the SciSciNet-v2 paper-field table to the paper IDs in the
analysis sample and then joins the official field dictionary. Only Level-0 and
Level-1 memberships are retained.

The join is configured with a bounded memory limit and a temporary directory so
that large intermediate partitions can spill to disk when necessary.

The remainder of the notebook uses Polars.

### Restarting an interrupted field join

If a previous run was interrupted during the large DuckDB join, delete the
partial file before rerunning the next cell:

```text
data/sciscinet_field_reconstruction_v2/
    analysis_matched_sciscinet_v2_fields.parquet
```
The notebook also checks the output metadata and automatically removes an
incomplete or incompatible file.


In [ ]:
remove_invalid_parquet(
    MATCHED_LINKS_CACHE,
    {"paperid", "fieldid", "display_name", "level"},
)

if FORCE_REBUILD_MATCHED_LINKS and MATCHED_LINKS_CACHE.exists():
    MATCHED_LINKS_CACHE.unlink()

if not parquet_valid(
    MATCHED_LINKS_CACHE,
    {"paperid", "fieldid", "display_name", "level"},
):
    t0 = perf_counter()

    # Clean up any partial output from an interrupted/stalled previous run.
    if MATCHED_LINKS_CACHE.exists():
        MATCHED_LINKS_CACHE.unlink()

    print("Building canonical paper↔field match with DuckDB")
    print("  Memory limit:", DUCKDB_MEMORY_LIMIT)
    print("  Threads:", DUCKDB_THREADS)
    print("  Temp spill directory:", DUCKDB_TEMP_DIR.resolve())
    print("  Output:", MATCHED_LINKS_CACHE.resolve())

    con = duckdb.connect()

    # Keep this join bounded well below the machine's 64 GB RAM.
    con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
    con.execute(f"SET threads={DUCKDB_THREADS}")
    con.execute(
        "SET temp_directory=?",
        [str(DUCKDB_TEMP_DIR.resolve())],
    )

    # Optional: preserve insertion order is unnecessary and can cost memory.
    try:
        con.execute("SET preserve_insertion_order=false")
    except Exception:
        pass

    # DuckDB reads the already-small target ID Parquet plus the public
    # SciSciNet-v2 Parquet files directly. No Python-side materialization of
    # the 33M-ID target table is required.
    target_path = str(TARGET_PAPERS_CACHE.resolve()).replace("\\", "/")
    pf_path = str(Path(PAPER_FIELDS_PATH).resolve()).replace("\\", "/")
    fields_path = str(Path(FIELDS_PATH).resolve()).replace("\\", "/")
    out_path = str(MATCHED_LINKS_CACHE.resolve()).replace("\\", "/")

    # Use the schema-resolved public column names discovered above.
    sql = f"""
    COPY (
        SELECT
            CAST(pf."{PF_PAPER_ID}" AS VARCHAR) AS paperid,
            CAST(pf."{PF_FIELD_ID}" AS VARCHAR) AS fieldid,
            CAST(f."{F_DISPLAY_NAME}" AS VARCHAR) AS display_name,
            CAST(f."{F_LEVEL}" AS INTEGER) AS level
        FROM read_parquet('{pf_path}') AS pf
        INNER JOIN read_parquet('{target_path}') AS t
            ON CAST(pf."{PF_PAPER_ID}" AS VARCHAR) = t.paperid
        INNER JOIN read_parquet('{fields_path}') AS f
            ON CAST(pf."{PF_FIELD_ID}" AS VARCHAR)
             = CAST(f."{F_FIELD_ID}" AS VARCHAR)
        WHERE CAST(f."{F_LEVEL}" AS INTEGER) IN (0, 1)
    )
    TO '{out_path}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD,
        ROW_GROUP_SIZE 100000
    )
    """

    try:
        con.execute(sql)
    finally:
        con.close()

    if not parquet_valid(
        MATCHED_LINKS_CACHE,
        {"paperid", "fieldid", "display_name", "level"},
    ):
        raise RuntimeError(
            "DuckDB join finished but output Parquet failed validation."
        )

    print(
        f"DuckDB matched-link join completed in "
        f"{(perf_counter() - t0) / 60:.2f} minutes."
    )

    matched_audit = (
        pl.scan_parquet(MATCHED_LINKS_CACHE)
        .select([
            pl.len().alias("rows"),
            pl.col("paperid").n_unique().alias("unique_papers"),
            pl.col("fieldid").n_unique().alias("unique_fieldids"),
            pl.col("level").min().alias("min_level"),
            pl.col("level").max().alias("max_level"),
        ])
        .collect()
    )

    print("\nMatched-link audit:")
    display(matched_audit.to_pandas())

else:
    print("Reusing:", MATCHED_LINKS_CACHE.resolve())

## 10. Create DISTINCT Level-0 and Level-1 long membership files

This reproduces the GitHub `ARRAY_AGG(DISTINCT ...)` semantics.

The Level-0 long file is the preferred handoff to the manuscript-analysis
notebook.

In [ ]:
def build_long_membership(level, output_path, output_column, force=False):
    remove_invalid_parquet(
        output_path,
        {"paperid", output_column},
    )

    if force and output_path.exists():
        output_path.unlink()

    if parquet_valid(output_path, {"paperid", output_column}):
        print("Reusing:", output_path.resolve())
        return

    (
        pl.scan_parquet(MATCHED_LINKS_CACHE)
        .filter(pl.col("level") == level)
        .select([
            "paperid",
            pl.col("display_name").alias(output_column),
        ])
        .unique(
            subset=["paperid", output_column],
            maintain_order=False,
        )
        .sink_parquet(
            output_path,
            compression="zstd",
            compression_level=3,
            statistics=True,
            maintain_order=False,
        )
    )

    print("Created:", output_path.resolve())


build_long_membership(
    level=0,
    output_path=LEVEL0_LONG,
    output_column="level_0_field_name",
    force=FORCE_REBUILD_OUTPUTS,
)

build_long_membership(
    level=1,
    output_path=LEVEL1_LONG,
    output_column="level_1_field_name",
    force=FORCE_REBUILD_OUTPUTS,
)

## 11. Create one-row-per-paper field arrays

In [ ]:
remove_invalid_parquet(
    PAPER_ARRAYS,
    {
        "paperid",
        "level_0_field_names",
        "level_1_field_names",
        "n_level_0_fields",
        "n_level_1_fields",
    },
)

if FORCE_REBUILD_OUTPUTS and PAPER_ARRAYS.exists():
    PAPER_ARRAYS.unlink()

if not parquet_valid(
    PAPER_ARRAYS,
    {
        "paperid",
        "level_0_field_names",
        "level_1_field_names",
        "n_level_0_fields",
        "n_level_1_fields",
    },
):
    level0_arrays = (
        pl.scan_parquet(LEVEL0_LONG)
        .group_by("paperid")
        .agg(
            pl.col("level_0_field_name")
            .sort()
            .alias("level_0_field_names")
        )
    )

    level1_arrays = (
        pl.scan_parquet(LEVEL1_LONG)
        .group_by("paperid")
        .agg(
            pl.col("level_1_field_name")
            .sort()
            .alias("level_1_field_names")
        )
    )

    (
        pl.scan_parquet(TARGET_PAPERS_CACHE)
        .select("paperid")
        .join(level0_arrays, on="paperid", how="left")
        .join(level1_arrays, on="paperid", how="left")
        .with_columns([
            pl.col("level_0_field_names")
            .list.len()
            .fill_null(0)
            .cast(pl.UInt16)
            .alias("n_level_0_fields"),

            pl.col("level_1_field_names")
            .list.len()
            .fill_null(0)
            .cast(pl.UInt16)
            .alias("n_level_1_fields"),
        ])
        .sink_parquet(
            PAPER_ARRAYS,
            compression="zstd",
            compression_level=3,
            statistics=True,
            maintain_order=False,
        )
    )

    print("Created:", PAPER_ARRAYS.resolve())
else:
    print("Reusing:", PAPER_ARRAYS.resolve())

## 12. Validate the reconstruction

In [ ]:
array_validation = (
    pl.scan_parquet(PAPER_ARRAYS)
    .select([
        pl.len().alias("total_papers"),

        (pl.col("n_level_0_fields") > 0)
        .sum()
        .alias("papers_with_level_0_fields"),

        (pl.col("n_level_1_fields") > 0)
        .sum()
        .alias("papers_with_level_1_fields"),

        (
            (pl.col("n_level_0_fields") == 0)
            & (pl.col("n_level_1_fields") == 0)
        )
        .sum()
        .alias("papers_without_fields"),

        pl.col("n_level_0_fields")
        .mean()
        .alias("avg_level_0_fields_per_paper"),

        pl.col("n_level_1_fields")
        .mean()
        .alias("avg_level_1_fields_per_paper"),

        (pl.col("n_level_0_fields") > 1)
        .sum()
        .alias("papers_with_multiple_level_0_fields"),

        pl.col("n_level_0_fields")
        .max()
        .alias("max_level_0_fields_per_paper"),
    ])
    .collect()
)

display(array_validation.to_pandas())

validation_stats = array_validation.row(0, named=True)

assert (
    validation_stats["total_papers"]
    == target_stats["unique_paperids"]
)

level0_validation = (
    pl.scan_parquet(LEVEL0_LONG)
    .select([
        pl.len().alias("membership_rows"),

        pl.struct([
            "paperid",
            "level_0_field_name",
        ])
        .n_unique()
        .alias("unique_pairs"),

        pl.col("level_0_field_name")
        .n_unique()
        .alias("unique_level_0_names"),
    ])
    .collect()
)

display(level0_validation.to_pandas())

l0v = level0_validation.row(0, named=True)

assert (
    l0v["membership_rows"] == l0v["unique_pairs"]
), "Duplicate Level-0 paper-field pairs remain."

assert l0v["unique_level_0_names"] == 19, (
    f"Expected 19 Level-0 field names; "
    f"found {l0v['unique_level_0_names']}."
)

observed_level0 = set(
    pl.scan_parquet(LEVEL0_LONG)
    .select("level_0_field_name")
    .unique()
    .collect()["level_0_field_name"]
    .to_list()
)

canonical_level0 = set(
    level0_lookup["level_0_field_name"].to_list()
)

missing = sorted(canonical_level0 - observed_level0)
unexpected = sorted(observed_level0 - canonical_level0)

print("Missing canonical Level-0 names:", missing)
print("Unexpected Level-0 names:", unexpected)

assert not missing
assert not unexpected

print("\nPASS: canonical Level-0 reconstruction validated.")

## 13. Level-0 membership frequencies

In [ ]:
level0_counts = (
    pl.scan_parquet(LEVEL0_LONG)
    .group_by("level_0_field_name")
    .agg(pl.len().alias("paper_memberships"))
    .sort("paper_memberships", descending=True)
    .collect()
)

display(level0_counts.to_pandas())
level0_counts.write_csv(LEVEL0_COUNTS_CSV)

## 14. Number of Level-0 memberships per paper

In [ ]:
level0_multiplicity = (
    pl.scan_parquet(PAPER_ARRAYS)
    .group_by("n_level_0_fields")
    .agg(pl.len().alias("papers"))
    .sort("n_level_0_fields")
    .collect()
    .with_columns(
        (
            pl.col("papers")
            / pl.col("papers").sum()
            * 100
        ).alias("percent_of_papers")
    )
)

display(level0_multiplicity.to_pandas())
level0_multiplicity.write_csv(MULTIPLICITY_CSV)

## 15. Diagnostic comparison with the analysis CSV's scalar `field_name`

This diagnostic is not used to construct manuscript field membership. It asks
whether the scalar `field_name` stored in the analysis CSV is also present
among each paper's canonical SciSciNet-v2 Level-0 or Level-1 memberships.

In [ ]:
scalar = (
    pl.scan_parquet(TARGET_PAPERS_CACHE)
    .filter(pl.col("scalar_field_name").is_not_null())
)

scalar_total = scalar.select(pl.len()).collect().item()

scalar_level0_match = (
    scalar
    .join(
        pl.scan_parquet(LEVEL0_LONG),
        left_on=["paperid", "scalar_field_name"],
        right_on=["paperid", "level_0_field_name"],
        how="semi",
    )
    .select(pl.len())
    .collect()
    .item()
)

scalar_level1_match = (
    scalar
    .join(
        pl.scan_parquet(LEVEL1_LONG),
        left_on=["paperid", "scalar_field_name"],
        right_on=["paperid", "level_1_field_name"],
        how="semi",
    )
    .select(pl.len())
    .collect()
    .item()
)

scalar_diag = pd.DataFrame([{
    "papers_with_scalar_field_name": scalar_total,
    "scalar_matches_level_0_membership": scalar_level0_match,
    "scalar_matches_level_0_pct": (
        100 * scalar_level0_match / scalar_total
        if scalar_total else None
    ),
    "scalar_matches_level_1_membership": scalar_level1_match,
    "scalar_matches_level_1_pct": (
        100 * scalar_level1_match / scalar_total
        if scalar_total else None
    ),
}])

display(scalar_diag)
scalar_diag.to_csv(SCALAR_DIAGNOSTIC_CSV, index=False)

## 16. Save a self-contained provenance manifest

In [ ]:
manifest = {
    "purpose": (
        "Reconstruct SciSciNet-v2 Level-0/Level-1 field memberships "
        "used by the Beginner's Charm manuscript pipeline."
    ),
    "analysis_input_csv": str(DATA_PATH.resolve()),
    "analysis_paper_id_system": "OpenAlex/SciSciNet-v2 W... string IDs",
    "large_join_engine": "DuckDB",
    "duckdb_memory_limit": DUCKDB_MEMORY_LIMIT,
    "duckdb_threads": DUCKDB_THREADS,
    "duckdb_temp_directory": str(DUCKDB_TEMP_DIR.resolve()),
    "github_reference": (
        "https://github.com/cssai-research/"
        "beginners_charm/blob/main/prepare_disruption_tables.py"
    ),
    "sciscinet_repo_id": REPO_ID,
    "huggingface_authenticated_account": hf_username,
    "huggingface_token_persisted": False,
    "sciscinet_requested_revision": REQUESTED_REVISION,
    "sciscinet_resolved_revision": RESOLVED_REVISION,
    "source_files": {
        "fields": str(Path(FIELDS_PATH).resolve()),
        "paper_fields": str(Path(PAPER_FIELDS_PATH).resolve()),
    },
    "resolved_schema": {
        "fields_fieldid": F_FIELD_ID,
        "fields_display_name": F_DISPLAY_NAME,
        "fields_level": F_LEVEL,
        "paperfields_paperid": PF_PAPER_ID,
        "paperfields_fieldid": PF_FIELD_ID,
    },
    "target_dataset": {
        "non_null_rows_used": int(target_stats["rows"]),
        "unique_non_null_paperids": int(target_stats["unique_paperids"]),
        "raw_null_paperids_excluded": int(raw_id_audit["raw_null_paperids"]),
    },
    "field_dictionary": {
        "level_0_fields": int(n_level0),
        "level_1_fields": int(n_level1),
        "level_0_names": sorted(canonical_level0),
    },
    "validation": {
        k: (float(v) if isinstance(v, float) else int(v))
        for k, v in validation_stats.items()
    },
    "outputs": {
        "preferred_level0_long": str(LEVEL0_LONG.resolve()),
        "level1_long": str(LEVEL1_LONG.resolve()),
        "paper_arrays": str(PAPER_ARRAYS.resolve()),
        "level0_lookup": str(LEVEL0_LOOKUP_CSV.resolve()),
        "scalar_field_diagnostic": str(SCALAR_DIAGNOSTIC_CSV.resolve()),
    },
    "later_analysis_contract": {
        "join_key": "paperid",
        "paperid_type": "string",
        "level0_field_column": "level_0_field_name",
        "multi_membership": True,
        "instruction": (
            "A paper may appear in multiple Level-0 fields. "
            "Retain all memberships in field-specific analyses."
        ),
    },
}

with open(MANIFEST_JSON, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Saved:", MANIFEST_JSON.resolve())

## 17. Final handoff summary

In [ ]:
print("=" * 78)
print("SCISCINET-v2 FIELD RECONSTRUCTION COMPLETE")
print("=" * 78)

print("\nAuthentication:")
print("  Hugging Face account:", hf_username)
print("  Token persisted in outputs: NO")

print("\nSource:")
print(" ", REPO_ID)
print("  revision:", RESOLVED_REVISION)

print("\nLarge join:")
print("  Engine: DuckDB")
print("  Memory limit:", DUCKDB_MEMORY_LIMIT)
print("  Temp spill directory:", DUCKDB_TEMP_DIR.resolve())

print("\nIdentifier system:")
print("  OpenAlex/SciSciNet-v2 W... paper IDs")

print("\nTarget analysis papers:")
print(f"  Non-null unique paper IDs used: {target_stats['unique_paperids']:,}")
print(
    f"  Raw rows excluded for null paperid: "
    f"{raw_id_audit['raw_null_paperids']:,}"
)

print("\nCanonical field dictionary:")
print("  Level-0 fields:", n_level0)
print("  Level-1 fields:", n_level1)

print("\nCoverage:")
print(
    "  Papers with >=1 Level-0 field:",
    f"{validation_stats['papers_with_level_0_fields']:,}",
)
print(
    "  Papers with >=1 Level-1 field:",
    f"{validation_stats['papers_with_level_1_fields']:,}",
)
print(
    "  Papers without either:",
    f"{validation_stats['papers_without_fields']:,}",
)
print(
    "  Mean Level-0 memberships/paper:",
    f"{validation_stats['avg_level_0_fields_per_paper']:.4f}",
)
print(
    "  Papers with >1 Level-0 membership:",
    f"{validation_stats['papers_with_multiple_level_0_fields']:,}",
)

print("\nPreferred input for the NEXT manuscript-analysis notebook:")
print(" ", LEVEL0_LONG.resolve())

print("\nRequired columns:")
print("  paperid               [String, W...]")
print("  level_0_field_name    [String]")

print("\nIMPORTANT:")
print("  This is a MULTI-MEMBERSHIP table.")
print("  Do not collapse each paper to one exclusive field.")

print("\nCanonical Level-0 names:")
for name in sorted(canonical_level0):
    print("  -", name)

print("\nDiagnostics/provenance:")
print(" ", SCALAR_DIAGNOSTIC_CSV.resolve())
print(" ", MANIFEST_JSON.resolve())

print("=" * 78)